In [1]:
import pandas as pd

In [2]:
clima_data_path = "/Users/antonio/Desktop/PT_Tortugas/Data/SERIE_MAESTRA_TAMUL_2018_2025_LOCAL.csv"
turtles_data_path = "/Users/antonio/Desktop/PT_Tortugas/Data/AnidacionesTamul_2018_2024_4.csv"

In [3]:
df_clima = pd.read_csv(clima_data_path)
df_tortugas = pd.read_csv(turtles_data_path)

In [4]:
df_clima.info()

<class 'pandas.DataFrame'>
RangeIndex: 70128 entries, 0 to 70127
Data columns (total 17 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   time_local              70128 non-null  str    
 1   time                    70128 non-null  str    
 2   tp_mm                   70123 non-null  float64
 3   ssrd_wm2                70123 non-null  float64
 4   swh                     70128 non-null  float64
 5   mwd                     70128 non-null  float64
 6   mwp                     70128 non-null  float64
 7   shts                    70128 non-null  float64
 8   tp                      5 non-null      float64
 9   ssrd                    5 non-null      float64
 10  t2m_celsius             70123 non-null  float64
 11  d2m_celsius             70123 non-null  float64
 12  stl1_celsius            70128 non-null  float64
 13  stl2_celsius            70128 non-null  float64
 14  stl3_celsius            70128 non-null  float64
 

In [5]:
df_tortugas.columns.tolist()

['id',
 'id_nido',
 'year',
 'especie',
 'fecha_recolecta',
 'hora_avistamiento',
 'estacion',
 'zona',
 'se_observo',
 'lscc_cm',
 'acc_cm',
 'hora_recolecta',
 'tamano_nidada',
 'hora_siembra',
 'huevos_sembrados',
 'fecha_prob_eclosion',
 'fecha_emergencia',
 'crias_vivas',
 'crias_muertas',
 'hsda',
 'hcda',
 'total_crias',
 'crias_deformes',
 'dias_incubacion',
 'GPS',
 'aleta_izq_marca_palace',
 'aleta_der_marca_palace',
 'no_serie_marca_palace',
 'recaptura_palace',
 'no_serie_recaptura',
 'aleta_izq_marca_externa',
 'aleta_der_marca_externa',
 'procedencia_marca_externa',
 'secuencia_fotografica_camara',
 'observaciones']

In [6]:
# Convertir clima
df_clima['time_local'] = pd.to_datetime(df_clima['time_local'])

In [7]:
# Convertir tortugas (Uniendo Fecha + Hora)
# Usamos errors='coerce' por si hay algún dato mal escrito en las horas
df_tortugas['fecha_hora_recolecta'] = pd.to_datetime(
    df_tortugas['fecha_recolecta'].astype(str) + ' ' + df_tortugas['hora_recolecta'].astype(str),
    errors='coerce'
)

In [8]:
# Eliminar filas donde la fecha no se pudo crear (si las hay)
df_tortugas = df_tortugas.dropna(subset=['fecha_hora_recolecta'])

Como los biólogos no recolectan nidos exactamente al minuto 00 de cada hora (que es como suele venir el clima de ERA5), usaremos merge_asof. Esta función busca la coincidencia más cercana en el tiempo.

In [9]:
# Ordenar indispensable
df_clima = df_clima.sort_values('time_local')
df_tortugas = df_tortugas.sort_values('fecha_hora_recolecta')

In [10]:
# Unir: Trae el clima a los datos de tortugas
df_final = pd.merge_asof(
    df_tortugas, 
    df_clima, 
    left_on='fecha_hora_recolecta', 
    right_on='time_local',
    direction='nearest' # Busca el registro horario más cercano
)

In [11]:
df_final = df_final.fillna('0')

In [12]:
df_final.head()

,id,id_nido,year,especie,fecha_recolecta,hora_avistamiento,estacion,zona,se_observo,lscc_cm,...,shts,tp,ssrd,t2m_celsius,d2m_celsius,stl1_celsius,stl2_celsius,stl3_celsius,wind_speed_10m_ms,wind_direction_10m_deg
0,1,2018_0001,2018,Dc,2018-04-07,10:30:00,82,0,Nido,0,...,0.540039,0,0,28.21224,21.31118,30.41714,27.29165,26.67010,7.685535,159.712405
1,2,2018_0002,2018,Cc,2018-05-15,09:35:00,68,C,Nido,0,...,0.927734,0,0,28.42470,23.51620,30.30355,27.14767,27.34194,4.501407,145.910290
2,3,2018_0003,2018,Cc,2018-05-16,09:50:00,65,C,Nido,0,...,0.984741,0,0,26.40423,22.84530,27.26360,26.75836,27.31060,4.218686,130.911774
3,4,2018_0004,2018,Cc,2018-05-19,09:30:00,29,C,Nido,0,...,1.262816,0,0,27.77828,22.32260,28.24697,27.11508,27.20940,6.574548,108.577712
4,5,2018_0005,2018,Cc,2018-05-19,09:50:00,28,C,Nido,0,...,1.262816,0,0,27.77828,22.32260,28.24697,27.11508,27.20940,6.574548,108.577712


In [13]:
 df_final.tail()

,id,id_nido,year,especie,fecha_recolecta,hora_avistamiento,estacion,zona,se_observo,lscc_cm,...,shts,tp,ssrd,t2m_celsius,d2m_celsius,stl1_celsius,stl2_celsius,stl3_celsius,wind_speed_10m_ms,wind_direction_10m_deg
4877,4878,2024_0600,2024,Cm,2024-10-01,08:00:00,60,C,Nido,0,...,0.562500,0,0,26.45278,24.99307,26.88790,27.56375,28.53127,5.549951,129.106207
4878,4879,2024_0601,2024,Cm,2024-10-04,07:00:00,67,B,Nido,0,...,0.832922,0,0,26.38300,24.92056,26.59518,27.48916,28.37194,5.578956,120.389587
4879,4880,2024_0602,2024,Cm,2024-10-07,03:30:00,50,B,Nido,0,...,0.297173,0,0,27.02554,25.22314,26.82060,27.70367,28.16970,4.774450,199.996389
4880,4881,2024_0603,2024,Cm,2024-10-15,12:00:00,30,C,Nido,0,...,0.637426,0,0,29.22585,23.94700,29.65457,27.43680,27.98104,3.151457,64.620497
4881,4882,2024_0604,2024,Cm,2024-11-01,10:43:00,67,B,Nido,0,...,1.579102,0,0,27.75814,22.47756,27.67587,26.72793,27.25704,6.864099,69.760214


In [14]:
df_final[["id_nido","estacion","zona","GPS"]].head()

,id_nido,estacion,zona,GPS
0,2018_0001,82,0,0
1,2018_0002,68,C,0
2,2018_0003,65,C,0
3,2018_0004,29,C,0
4,2018_0005,28,C,0


In [15]:
#df_final.to_csv("/Users/antonio/Desktop/PT_Tortugas/Data/Data_Joined.csv")

In [16]:
df_final.columns.tolist()

['id',
 'id_nido',
 'year',
 'especie',
 'fecha_recolecta',
 'hora_avistamiento',
 'estacion',
 'zona',
 'se_observo',
 'lscc_cm',
 'acc_cm',
 'hora_recolecta',
 'tamano_nidada',
 'hora_siembra',
 'huevos_sembrados',
 'fecha_prob_eclosion',
 'fecha_emergencia',
 'crias_vivas',
 'crias_muertas',
 'hsda',
 'hcda',
 'total_crias',
 'crias_deformes',
 'dias_incubacion',
 'GPS',
 'aleta_izq_marca_palace',
 'aleta_der_marca_palace',
 'no_serie_marca_palace',
 'recaptura_palace',
 'no_serie_recaptura',
 'aleta_izq_marca_externa',
 'aleta_der_marca_externa',
 'procedencia_marca_externa',
 'secuencia_fotografica_camara',
 'observaciones',
 'fecha_hora_recolecta',
 'time_local',
 'time',
 'tp_mm',
 'ssrd_wm2',
 'swh',
 'mwd',
 'mwp',
 'shts',
 'tp',
 'ssrd',
 't2m_celsius',
 'd2m_celsius',
 'stl1_celsius',
 'stl2_celsius',
 'stl3_celsius',
 'wind_speed_10m_ms',
 'wind_direction_10m_deg']

In [17]:
df_2024 = df_final[df_final['GPS'] != "0"].reset_index(drop=True)

In [18]:
df_estacion = df_2024[df_2024["estacion"] == 28]
df_estacion[["id_nido","estacion","zona","GPS"]]

,id_nido,estacion,zona,GPS
138,2024_0380,28,C,N20 59.900 W86 49.385
181,2024_0426,28,C,N20 59.899 W86 49.384


In [19]:
df_2024["GPS"].tail()

354    N20 58.165 W86 49.853
355    N20 58.416 W86 49.852
356    N20 59.795 W86 49.410
357    N20 59.795 W86 49.410
358    N20 58.968 W86 49.798
Name: GPS, dtype: str

In [20]:
df_2024.info()

<class 'pandas.DataFrame'>
RangeIndex: 359 entries, 0 to 358
Data columns (total 53 columns):
 #   Column                        Non-Null Count  Dtype         
---  ------                        --------------  -----         
 0   id                            359 non-null    int64         
 1   id_nido                       359 non-null    str           
 2   year                          359 non-null    int64         
 3   especie                       359 non-null    str           
 4   fecha_recolecta               359 non-null    str           
 5   hora_avistamiento             359 non-null    str           
 6   estacion                      359 non-null    int64         
 7   zona                          359 non-null    str           
 8   se_observo                    359 non-null    str           
 9   lscc_cm                       359 non-null    int64         
 10  acc_cm                        359 non-null    int64         
 11  hora_recolecta                359 non-null 

In [21]:
#df_2024.to_csv("/Users/antonio/Desktop/PT_Tortugas/Data/Data_2024.csv")

# Unir coordenadas desde el 2020

In [22]:
import pandas as pd
import numpy as np

# 1. Crear el diccionario de referencia (Catálogo de 2024)
# Agrupamos por zona y estación y tomamos el primer valor de GPS que encontremos
df_referencia = df_final[df_final['year'] == 2024].dropna(subset=['GPS'])
df_referencia = df_referencia.groupby(['zona', 'estacion'])['GPS'].first().reset_index()

df_referencia.head()

,zona,estacion,GPS
0,B,10,N21 00.853 W86 49.031
1,B,12,N21 00.725 W86 49.073
2,B,15,N21 00.563 W86 49.151
3,B,16,N21 00.562 W86 49.151
4,B,17,N21 00.496 W86 49.185


In [23]:
# 2. Separar los datos de 2020 en adelante
df_principal = df_final[df_final['year'] >= 2018].copy()
df_principal['GPS'] = df_principal['GPS'].replace({0: np.nan, '0': np.nan})
# 3. Unir la referencia como una columna temporal
df_principal = df_principal.merge(
    df_referencia, 
    on=['zona', 'estacion'], 
    how='left', 
    suffixes=('', '_ref')
)

df_principal[['GPS_ref', 'year']].head()

,GPS_ref,year
0,NaN,2018
1,NaN,2018
2,N20 57.884 W86 49.894,2018
3,0,2018
4,0,2018


In [24]:
# 4. Rellenar los nulos de la columna 'GPS' original con los de la columna '_ref'
df_principal['GPS'] = df_principal['GPS'].fillna(df_principal['GPS_ref'])

df_principal[['GPS', 'year']].head()

,GPS,year
0,NaN,2018
1,NaN,2018
2,N20 57.884 W86 49.894,2018
3,0,2018
4,0,2018


In [25]:
# 5. Limpiar: eliminar la columna temporal y continuar
df_principal = df_principal.drop(columns=['GPS_ref'])

# Si quieres unirlo con los años anteriores (< 2020):
df_resto = df_final[df_final['year'] < 2018]
df_resultado = pd.concat([df_resto, df_principal], ignore_index=True)

In [26]:
df_resultado.info()

<class 'pandas.DataFrame'>
RangeIndex: 4882 entries, 0 to 4881
Data columns (total 53 columns):
 #   Column                        Non-Null Count  Dtype         
---  ------                        --------------  -----         
 0   id                            4882 non-null   int64         
 1   id_nido                       4882 non-null   str           
 2   year                          4882 non-null   int64         
 3   especie                       4882 non-null   str           
 4   fecha_recolecta               4882 non-null   str           
 5   hora_avistamiento             4882 non-null   str           
 6   estacion                      4882 non-null   int64         
 7   zona                          4882 non-null   str           
 8   se_observo                    4882 non-null   str           
 9   lscc_cm                       4882 non-null   int64         
 10  acc_cm                        4882 non-null   int64         
 11  hora_recolecta                4882 non-nu

In [27]:
df_since2020 = df_resultado[df_resultado['year'] >= 2018]

In [28]:
df_since2020 = df_resultado[df_resultado['GPS'] != '0']

In [29]:
df_since2020 = df_since2020.dropna()

In [30]:
df_since2020[["year","GPS"]].head()

,year,GPS
2,2018,N20 57.884 W86 49.894
7,2018,N20 58.213 W86 49.839
8,2018,N20 58.258 W86 49.839
19,2018,N20 58.213 W86 49.839
22,2018,N20 58.213 W86 49.839


In [31]:
df_since2020.info()

<class 'pandas.DataFrame'>
Index: 725 entries, 2 to 4881
Data columns (total 53 columns):
 #   Column                        Non-Null Count  Dtype         
---  ------                        --------------  -----         
 0   id                            725 non-null    int64         
 1   id_nido                       725 non-null    str           
 2   year                          725 non-null    int64         
 3   especie                       725 non-null    str           
 4   fecha_recolecta               725 non-null    str           
 5   hora_avistamiento             725 non-null    str           
 6   estacion                      725 non-null    int64         
 7   zona                          725 non-null    str           
 8   se_observo                    725 non-null    str           
 9   lscc_cm                       725 non-null    int64         
 10  acc_cm                        725 non-null    int64         
 11  hora_recolecta                725 non-null    s

In [32]:
df_since2020.to_csv("/Users/antonio/Desktop/PT_Tortugas/Data/datawithgpssince2018.csv")